In [1]:
from collections import Counter
import math
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = {
    "D1": "information retrieval retrieval system".split(),
    "D2": "information system system system".split(),
    "D3": "retrieval model system".split(),
    "D4": "information retrieval model model".split(),
}

vocabulary = ["information", "retrieval", "system", "model"]
doc_ids = list(corpus.keys())
n_documents = len(corpus)

# Part 1: Term Frequency
* Raw TF counts how many times term $t$ appears in document $d$: $tf(t, d)$.
* Normalized TF divides raw TF by the total token length of document $d$: $tf\_norm(t, d) = \frac{tf(t, d)}{\vert{}d\vert{}}$.

In [2]:
def compute_tf(document: list[str]) -> dict[str, int]:
    """Compute raw term frequencies for a tokenized document.

    Args:
        document: List of tokens in a single document.

    Returns:
        Dictionary mapping each term to its raw occurrence count.
    """
    counts = Counter(document)
    return {term: counts.get(term, 0) for term in vocabulary}


def build_raw_tf_matrix(
    docs: dict[str, list[str]], vocab: list[str]
) -> pd.DataFrame:
    """Build the raw TF matrix with documents as rows and fixed vocabulary as columns."""
    rows = []
    for doc_id, tokens in docs.items():
        tf_map = compute_tf(tokens)
        rows.append([tf_map[term] for term in vocab])
    return pd.DataFrame(rows, index=list(docs.keys()), columns=vocab)


def compute_normalized_tf(document: list[str]) -> dict[str, float]:
    """Compute length-normalized term frequencies for a tokenized document.

    Args:
        document: List of tokens in a single document.

    Returns:
        Dictionary mapping each term to tf(t, d) / total_terms_in_d.
    """
    total_terms = len(document)
    if total_terms == 0:
        return {term: 0.0 for term in vocabulary}
    raw_tf = compute_tf(document)
    return {term: raw_tf[term] / total_terms for term in vocabulary}


def build_normalized_tf_matrix(
    docs: dict[str, list[str]], vocab: list[str]
) -> pd.DataFrame:
    """Build the normalized TF matrix with documents as rows and fixed vocabulary as columns."""
    rows = []
    for doc_id, tokens in docs.items():
        norm_tf_map = compute_normalized_tf(tokens)
        rows.append([norm_tf_map[term] for term in vocab])
    return pd.DataFrame(rows, index=list(docs.keys()), columns=vocab)


def most_frequent_terms(raw_tf_df: pd.DataFrame) -> dict[str, list[str]]:
    """Return the highest-frequency term(s) per document, preserving ties."""
    top_terms = {}
    for doc_id, row in raw_tf_df.iterrows():
        max_val = row.max()
        top_terms[doc_id] = [term for term, val in row.items() if val == max_val]
    return top_terms


tf_raw_df = build_raw_tf_matrix(corpus, vocabulary)
tf_norm_df = build_normalized_tf_matrix(corpus, vocabulary)
top_tf_by_doc = most_frequent_terms(tf_raw_df)

print("Raw TF Matrix:\n", tf_raw_df, "\n")
print("Normalized TF Matrix:\n", tf_norm_df.round(4), "\n")
print("Most frequent term(s) per document:", top_tf_by_doc)

Raw TF Matrix:
     information  retrieval  system  model
D1            1          2       1      0
D2            1          0       3      0
D3            0          1       1      1
D4            1          1       0      2 

Normalized TF Matrix:
     information  retrieval  system   model
D1         0.25     0.5000  0.2500  0.0000
D2         0.25     0.0000  0.7500  0.0000
D3         0.00     0.3333  0.3333  0.3333
D4         0.25     0.2500  0.0000  0.5000 

Most frequent term(s) per document: {'D1': ['retrieval'], 'D2': ['system'], 'D3': ['retrieval', 'system', 'model'], 'D4': ['model']}


# Explanation
**Raw TF Matrix:**
* D1 (4 terms): information=1, retrieval=2, system=1, model=0 $\rightarrow$ most frequent: retrieval (2).
* D2 (4 terms): information=1, retrieval=0, system=3, model=0 $\rightarrow$ most frequent: system (3).
* D3 (3 terms): information=0, retrieval=1, system=1, model=1 $\rightarrow$ tie between retrieval, system, model (1 each).
* D4 (4 terms): information=1, retrieval=1, system=0, model=2 $\rightarrow$ most frequent: model (2).

**Normalized TF Matrix:**

Because D1, D2, and D4 each have 4 tokens, dividing by 4 scales counts by 0.25 (e.g., system in D2 is $3/4 = 0.75$). Because D3 has 3 tokens, each occurrence in D3 is $1/3 \approx 0.3333$.

---

# Part 2: Document Frequency and IDF
* DF counts in how many documents a term appears at least once ($df(t) \in [1, N]$ here).
* IDF-1: $\text{idf}_1(t) = \log_{10}\left(\frac{N}{df(t)}\right)$.
* IDF-2 (smoothed): $\text{idf}_2(t) = \log_{10}\left(1 + \frac{N}{df(t)}\right)$.

In [3]:
def compute_df(docs: dict[str, list[str]], vocab: list[str]) -> dict[str, int]:
    """Compute document frequency df(t) for each term in the vocabulary.

    Each term contributes at most once per document.
    """
    df_counts = {term: 0 for term in vocab}
    for tokens in docs.values():
        unique_tokens = set(tokens)
        for term in vocab:
            if term in unique_tokens:
                df_counts[term] += 1
    return df_counts


def compute_idf_1(df: dict[str, int], n_docs: int) -> dict[str, float]:
    """Compute IDF-1: log10(N / df(t))."""
    return {
        term: math.log10(n_docs / count) if count > 0 else 0.0
        for term, count in df.items()
    }


def compute_idf_2(df: dict[str, int], n_docs: int) -> dict[str, float]:
    """Compute IDF-2 (smoothed): log10(1 + N / df(t))."""
    return {
        term: math.log10(1.0 + (n_docs / count)) if count > 0 else 0.0
        for term, count in df.items()
    }


df_map = compute_df(corpus, vocabulary)
idf1_map = compute_idf_1(df_map, n_documents)
idf2_map = compute_idf_2(df_map, n_documents)

df_idf_table = pd.DataFrame({
    "term": vocabulary,
    "DF": [df_map[t] for t in vocabulary],
    "IDF-1": [idf1_map[t] for t in vocabulary],
    "IDF-2": [idf2_map[t] for t in vocabulary],
})

print(df_idf_table.to_string(index=False))

       term  DF    IDF-1    IDF-2
information   3 0.124939 0.367977
  retrieval   3 0.124939 0.367977
     system   3 0.124939 0.367977
      model   2 0.301030 0.477121


# Key observations:

* Highest IDF under both conventions: model ($DF = 2$, rarest in the corpus).
* Lowest IDF under both conventions: tie among information, retrieval, and system ($DF = 3$, most widespread).
* When a term appears in every document ($DF = N$): Under IDF-1, $N / df(t) = 1$, so $\log_{10}(1) = 0$ (zero weight regardless of TF). Under IDF-2, $1 + N / df(t) = 2$, so $\log_{10}(2) \approx 0.3010 > 0$, preserving a non-zero baseline weight. Overall, IDF-2 is strictly larger than IDF-1 and compresses relative ratios between rare and common terms because of the added 1 inside the logarithm.

---

# Part 3: TF-IDF Weighting Matrix
We compute $w(t, d) = tf(t, d) \times idf_1(t)$ using raw TF (main matrix) and using normalized TF (second matrix).

In [4]:
def compute_tfidf_matrix(
    tf_matrix: pd.DataFrame, idf_map: dict[str, float]
) -> pd.DataFrame:
    """Compute TF-IDF matrix w(t, d) = tf(t, d) * idf(t) column-wise."""
    idf_series = pd.Series(idf_map)[tf_matrix.columns]
    return tf_matrix.multiply(idf_series, axis="columns")


def highest_weighted_terms(tfidf_df: pd.DataFrame) -> dict[str, list[tuple[str, float]]]:
    """Identify the highest-weighted term(s) and their scores in each document."""
    result = {}
    for doc_id, row in tfidf_df.iterrows():
        max_score = row.max()
        winners = [
            (term, round(float(score), 4))
            for term, score in row.items()
            if math.isclose(float(score), float(max_score), rel_tol=1e-9, abs_tol=1e-12)
        ]
        result[doc_id] = winners
    return result


tfidf_raw_df = compute_tfidf_matrix(tf_raw_df, idf1_map)
tfidf_norm_df = compute_tfidf_matrix(tf_norm_df, idf1_map)

print("TF-IDF Matrix (Raw TF * IDF-1):\n", tfidf_raw_df.round(4), "\n")
print("TF-IDF Matrix (Normalized TF * IDF-1):\n", tfidf_norm_df.round(4), "\n")
print("Highest-weighted term(s) (Raw TF):\n", highest_weighted_terms(tfidf_raw_df))
print("Highest-weighted term(s) (Normalized TF):\n", highest_weighted_terms(tfidf_norm_df))

TF-IDF Matrix (Raw TF * IDF-1):
     information  retrieval  system   model
D1       0.1249     0.2499  0.1249  0.0000
D2       0.1249     0.0000  0.3748  0.0000
D3       0.0000     0.1249  0.1249  0.3010
D4       0.1249     0.1249  0.0000  0.6021 

TF-IDF Matrix (Normalized TF * IDF-1):
     information  retrieval  system   model
D1       0.0312     0.0625  0.0312  0.0000
D2       0.0312     0.0000  0.0937  0.0000
D3       0.0000     0.0416  0.0416  0.1003
D4       0.0312     0.0312  0.0000  0.1505 

Highest-weighted term(s) (Raw TF):
 {'D1': [('retrieval', 0.2499)], 'D2': [('system', 0.3748)], 'D3': [('model', 0.301)], 'D4': [('model', 0.6021)]}
Highest-weighted term(s) (Normalized TF):
 {'D1': [('retrieval', 0.0625)], 'D2': [('system', 0.0937)], 'D3': [('model', 0.1003)], 'D4': [('model', 0.1505)]}


# Why high TF does not guarantee high TF-IDF:

TF-IDF is multiplicative. If a term occurs frequently across almost all documents, its document frequency $df(t)$ is large and $N / df(t)$ is small, driving $\text{idf}(t)$ down. A very low IDF can outweigh a high TF.

---

# Part 4: Verification with scikit-learn
When we call *TfidfVectorizer(sublinear_tf=False, smooth_idf=False, norm=None)* on pre-tokenized documents (passing a pass-through *tokenizer=lambda x: x* and *preprocessor=None* with *lowercase=False* so it does not re-tokenize or alter tokens):

In [5]:
# Join tokens with spaces so TfidfVectorizer with a whitespace/identity tokenizer sees exact tokens
joined_docs = [" ".join(corpus[doc_id]) for doc_id in doc_ids]

vectorizer = TfidfVectorizer(
    vocabulary=vocabulary,
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    sublinear_tf=False,
    smooth_idf=False,
    norm=None,
)

sklearn_tfidf = vectorizer.fit_transform(joined_docs).toarray()
sklearn_tfidf_df = pd.DataFrame(sklearn_tfidf, index=doc_ids, columns=vocabulary)

# scikit-learn's internal IDF formula when smooth_idf=False: ln(N / df) + 1
sklearn_idf = vectorizer.idf_
sklearn_idf_series = pd.Series(sklearn_idf, index=vocabulary)

print("scikit-learn IDF (ln(N/df) + 1):\n", sklearn_idf_series.round(4), "\n")
print("scikit-learn TF-IDF Matrix:\n", sklearn_tfidf_df.round(4), "\n")

# Compare D4 ('model') directly:
# Course IDF-1 for 'model': log10(4/2) = 0.30103; raw TF in D4 = 2 -> 0.60206
# sklearn IDF for 'model': ln(4/2) + 1 = 0.693147 + 1 = 1.693147; raw TF in D4 = 2 -> 3.386294
print("D4 'model' - Course raw TF-IDF:", round(tfidf_raw_df.loc["D4", "model"], 6))
print("D4 'model' - sklearn TF-IDF:   ", round(sklearn_tfidf_df.loc["D4", "model"], 6))

scikit-learn IDF (ln(N/df) + 1):
 information    1.2877
retrieval      1.2877
system         1.2877
model          1.6931
dtype: float64 

scikit-learn TF-IDF Matrix:
     information  retrieval  system   model
D1       1.2877     2.5754  1.2877  0.0000
D2       1.2877     0.0000  3.8630  0.0000
D3       0.0000     1.2877  1.2877  1.6931
D4       1.2877     1.2877  0.0000  3.3863 

D4 'model' - Course raw TF-IDF: 0.60206
D4 'model' - sklearn TF-IDF:    3.386294


# Why the Values Differ
Even with *sublinear_tf=False*, *smooth_idf=False*, and *norm=None* :
1. Logarithm base: The course uses base-10 ($\log_{10}$), whereas
2. scikit-learn uses the natural logarithm ($\ln$, base $e$).
Additive + 1 offset in scikit-learn: With smooth_idf=False, scikit-learn computes $\text{idf}(t) = \ln(N / df(t)) + 1$ (so a term appearing in all documents gets an IDF of 1.0, not 0.0), whereas the course's IDF-1 has no + 1 term.
3. Verified example (model in D4): Raw TF is 2 in both. Under the course's IDF-1, $\text{idf}_1(\text{model}) = \log_{10}(2) \approx 0.3010$, giving $2 \times 0.3010 = 0.6021$. Under scikit-learn, $\text{idf}_{\text{sk}}(\text{model}) = \ln(2) + 1 \approx 1.6931$, giving $2 \times 1.6931 = 3.3863$.

---

In [6]:
os.makedirs("results", exist_ok=True)
tf_raw_df.to_csv("results/tf matrix.csv")
tf_norm_df.to_csv("results/normalized tf matrix.csv")
df_idf_table.to_csv("results/df_idf_comparison.csv", index=False)
tfidf_raw_df.to_csv("results/tfidf_raw_matrix.csv")
tfidf_norm_df.to_csv("results/tfidf normalized matrix.csv")

# Analysis Questions
19. High TF $\neq$ always high TF-IDF: Because $w(t, d) = tf(t, d) \times idf(t)$, a high $tf(t, d)$ paired with a very low $idf(t)$ (e.g., a term in nearly every document) produces a small weight, while a moderate TF paired with a high IDF can produce a larger weight.
20. Effect of normalized TF across document lengths: Dividing by document length prevents longer documents from automatically having larger TF values solely because they contain more words, placing terms on a relative proportion scale ($[0, 1]$) across short and long documents.
21. Term in every document ($df(t) = N$): IDF-1 yields $\log_{10}(N/N) = \log_{10}(1) = 0$ (zeroing out the term), whereas IDF-2 yields $\log_{10}(1 + N/N) = \log_{10}(2) \approx 0.3010$.
22. High frequency, low IDF example: In a corpus of computing notes where "system" appears in almost every document and multiple times per document, its frequency is high but its IDF is low, which dampens its TF-IDF weight and prevents it from distinguishing specific documents.
23. Consistent vocabulary order: Keeping ["information", "retrieval", "system", "model"] fixed ensures every column aligns with the intended term across matrices, dot products/similarities, and comparisons with external tools.